# MLE — Train Arcade Game RL Agent on Colab GPU

Trains RL agents (PPO or Dreamer v3) on any MAME arcade game using GPU acceleration.

**Upload your ROM zip** (e.g. `frogger.zip`) when prompted.

In [ ]:
# 1. Install MAME + SDL + dependencies
!apt-get update -qq && apt-get install -y -qq mame xvfb > /dev/null 2>&1
!pip install -q gymnasium stable-baselines3 pillow wandb

# Start virtual display for SDL
import subprocess, os, shutil
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1x1x24'])
os.environ['DISPLAY'] = ':99'
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'
os.environ['XDG_RUNTIME_DIR'] = '/tmp'

# Add /usr/games to PATH (Ubuntu installs mame there)
if '/usr/games' not in os.environ.get('PATH', ''):
    os.environ['PATH'] = '/usr/games:' + os.environ['PATH']

# Verify MAME is found
mame_bin = shutil.which('mame')
if mame_bin:
    print(f'MAME found: {mame_bin}')
    !{mame_bin} -version 2>/dev/null || {mame_bin} -help 2>&1 | head -1
else:
    print('ERROR: MAME not found! Check apt install output above.')

In [ ]:
# 2. Clone MLE repo
!git clone https://github.com/lilwg/mle.git
%cd mle

In [ ]:
# 3. Upload ROM file
from google.colab import files
import os

uploaded = files.upload()

os.makedirs('roms', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'roms/{fname}')
    print(f'ROM: roms/{fname}')

rom_name = [f[:-4] for f in os.listdir('roms') if f.endswith('.zip')][0]
print(f'Game: {rom_name}')

In [ ]:
# 3b. Fix ROM compatibility
# Different MAME versions expect different ROM filenames.
# This cell checks and fixes mismatches automatically.
import subprocess, zipfile, re, shutil, tempfile

def find_mame_binary():
    """Find the MAME binary."""
    b = shutil.which('mame')
    if b:
        return b
    for p in ['/usr/games/mame', '/usr/bin/mame', '/usr/local/bin/mame', '/snap/bin/mame']:
        if os.path.isfile(p) and os.access(p, os.X_OK):
            return p
    raise FileNotFoundError('MAME not found')

def fix_rom_compat(roms_dir, game):
    """Check ROM compatibility and attempt to fix filename mismatches."""
    mame = find_mame_binary()
    rom_path = os.path.join(roms_dir, f'{game}.zip')
    if not os.path.exists(rom_path):
        print(f'ERROR: {rom_path} not found')
        return False

    # Run verifyroms to check compatibility
    result = subprocess.run(
        [mame, '-rompath', roms_dir, game, '-verifyroms'],
        capture_output=True, text=True, timeout=30
    )
    output = result.stdout + result.stderr

    if 'is good' in output.lower():
        print(f'ROM {game} is compatible!')
        return True

    # Find expected filenames that are NOT FOUND
    not_found = re.findall(r'(\S+)\s+NOT FOUND', output)
    if not not_found:
        not_found = re.findall(r'NOT FOUND\s*-\s*(\S+)', output)

    if not not_found:
        print(f'ROM check output:\n{output}')
        print('Could not parse missing files. ROM may be incompatible.')
        return False

    print(f'Missing ROM files: {not_found}')

    with zipfile.ZipFile(rom_path, 'r') as zf:
        existing = zf.namelist()
    print(f'Files in zip: {existing}')

    # Get expected ROM list from MAME's listxml
    result2 = subprocess.run(
        [mame, game, '-listxml'],
        capture_output=True, text=True, timeout=30
    )
    xml = result2.stdout

    expected_roms = {}
    for m in re.finditer(r'<rom name="([^"]+)"[^/]*(?:size="(\d+)")?[^/]*(?:crc="([^"]*)")?', xml):
        name, size, crc = m.group(1), m.group(2), m.group(3)
        expected_roms[name] = {'size': int(size) if size else 0, 'crc': crc or ''}

    if not expected_roms:
        print('Could not get expected ROM list from MAME.')
        return False

    print(f'\nExpected ROMs: {list(expected_roms.keys())}')

    # Build mapping: match by file size
    size_map = {}
    with zipfile.ZipFile(rom_path, 'r') as zf:
        for info in zf.infolist():
            sz = info.file_size
            size_map.setdefault(sz, []).append(info.filename)

    rename_map = {}
    for expected_name, props in expected_roms.items():
        if expected_name in existing:
            continue
        sz = props['size']
        candidates = size_map.get(sz, [])
        if len(candidates) == 1:
            rename_map[candidates[0]] = expected_name
            print(f'  {candidates[0]} -> {expected_name} (matched by size {sz})')
        elif len(candidates) > 1:
            best = None
            for c in candidates:
                c_base = c.rsplit('.', 1)[0].lower()
                e_base = expected_name.rsplit('.', 1)[0].lower()
                if c_base in e_base or e_base in c_base:
                    best = c
                    break
            if best:
                rename_map[best] = expected_name
                print(f'  {best} -> {expected_name} (matched by name+size)')
            else:
                print(f'  WARNING: {expected_name} (size {sz}) has {len(candidates)} candidates: {candidates}')

    if not rename_map:
        print('\nNo automatic fixes found. ROMs may be for a different MAME version.')
        return False

    # Repack zip with renamed files
    tmp_path = rom_path + '.tmp'
    with zipfile.ZipFile(rom_path, 'r') as zf_in:
        with zipfile.ZipFile(tmp_path, 'w') as zf_out:
            for info in zf_in.infolist():
                data = zf_in.read(info.filename)
                new_name = rename_map.get(info.filename, info.filename)
                zf_out.writestr(new_name, data)

    os.replace(tmp_path, rom_path)
    print(f'\nRepacked {rom_path} with {len(rename_map)} renamed files.')

    # Verify again
    result3 = subprocess.run(
        [mame, '-rompath', roms_dir, game, '-verifyroms'],
        capture_output=True, text=True, timeout=30
    )
    if 'is good' in (result3.stdout + result3.stderr).lower():
        print('ROM is now compatible!')
        return True
    else:
        print(f'Still failing:\n{result3.stdout + result3.stderr}')
        return False

fix_rom_compat('roms', rom_name)

In [ ]:
# 4. Quick test
import sys, os
sys.path.insert(0, '.')
os.environ['MLE_ROMS_PATH'] = 'roms'
from mle import MameEnv

env = MameEnv('roms', rom_name, {'_dummy': 0}, render=False, sound=False, throttle=False)
env.wait(100)
d = env.step()
print(f'MAME works! Got {len(d)} values')
env.close()
print('Ready to train!')

## Option A: PPO Training (stable-baselines3)
Fast to start, good for initial experiments. Uses CNN policy on pixels.

In [ ]:
# 5. Train with parallel MAME instances!
GAME = rom_name
TIMESTEPS = 2_000_000  # 2M steps
MODEL = 'ppo'
N_PARALLEL = 4          # 4 MAME instances = ~4x faster

save_name = f'{GAME}_{MODEL}_{TIMESTEPS//1000}k'
print(f'Training {GAME} with {MODEL}, {TIMESTEPS} steps, {N_PARALLEL} parallel envs')

!MLE_ROMS_PATH=roms python3 mle_general.py {GAME} --model {MODEL} --timesteps {TIMESTEPS} \
    --save {save_name} --parallel {N_PARALLEL}

In [ ]:
# 6. Download trained model
from google.colab import files
import glob
for f in glob.glob('*.zip'):
    if 'cheat' not in f:
        print(f'Downloading {f}...')
        files.download(f)

## Option B: Dreamer v3 Training (sheeprl)
World-model based RL — much better sample efficiency (~4500x vs PPO). Requires Python 3.11 venv.

In [ ]:
# 7. Install Python 3.11 + sheeprl for Dreamer v3
!apt-get install -y -qq python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1
!python3.11 -m venv /content/venv311
!source /content/venv311/bin/activate && pip install -q \
    'numpy<2' && \
    pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!source /content/venv311/bin/activate && pip install -q \
    sheeprl gymnasium pillow wandb opencv-python-headless
print("Python 3.11 venv with sheeprl ready!")

In [ ]:
# 8. Train Dreamer v3
GAME = rom_name
TIMESTEPS = 500_000

print(f'Training Dreamer v3 on {GAME} for {TIMESTEPS} steps...')
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "No GPU detected"

!source /content/venv311/bin/activate && cd /content/mle && \
    DISPLAY=:99 SDL_VIDEODRIVER=dummy SDL_AUDIODRIVER=dummy \
    MLE_ROMS_PATH=/content/mle/roms \
    python -u dreamer_train.py {GAME} --timesteps {TIMESTEPS}